# 06 — GPU Baselines on Colab (Task 3)

Run `DIPK`, `DrugGNN`, and `SimpleNeuralNetwork` from drevalpy's `MODEL_FACTORY` —
same pattern as notebook 05, but these train per-fold with early stopping (and,
for `DrugGNN`/`SimpleNeuralNetwork`, a real hyperparameter grid), so budget real
GPU time.

**Hyperparameter grid sizes** (read directly from each model's `hyperparameters.yaml`
in the installed drevalpy 1.5.1 — not assumed):
- `DIPK`: 1 combination (100 epochs + 100-epoch autoencoder pretrain) — `hyperparameter_tuning`
  on/off makes no practical difference here, the grid only has one entry.
- `DrugGNN`: 2 x 2 = 4 combinations (hidden_dim x dropout), 100 epochs each.
- `SimpleNeuralNetwork`: 4 combinations (units_per_layer), 100 max_epochs.

**Shares `RUN_ID` with notebook 05** — if 05 already ran a given split, this reuses
those exact folds (drevalpy loads saved splits from disk automatically). Run 05 first.

Per `NEXT_SESSION.md`: sanity-check runtime on a single split (**LCO**) before
committing to all four — that's structured as two separate steps below so you can
stop and look at the timing before continuing.

## Environment setup (Colab or local)

Detects Colab automatically. On Colab, mounts Drive and expects (or will create)
project data under `MyDrive/multiomics_project/` — same `BASE_PATH` convention as
`00_colab_setup.ipynb` from Session 2. If `data/GDSC2/GDSC2.csv` isn't there yet,
drevalpy will auto-download it from Zenodo on first use (slower, self-healing —
no need to manually upload first if you'd rather let it fetch fresh).

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q "drevalpy[xgboost]" torch-geometric
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/multiomics_project')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

## GPU check

Colab Runtime → Change runtime type → GPU, if this shows no GPU.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected — these models will be very slow on CPU. "
          "Runtime > Change runtime type > GPU, then re-run this cell.")

## Imports and config

In [ ]:
import time
from glob import glob

import pandas as pd
from scipy.stats import pearsonr

from drevalpy.datasets.loader import load_dataset
from drevalpy.experiment import drug_response_experiment
from drevalpy.models import MODEL_FACTORY

In [ ]:
DATA_DIR = BASE_PATH / "data"
RESULTS_DIR = BASE_PATH / "results"
RUN_ID = "session4_baselines"   # shared with 05_baselines_local.ipynb — see note above

N_CV_SPLITS = 5
RESPONSE_MEASURE = "LN_IC50"    # see notebook 05 for why this isn't the drevalpy default
HYPERPARAMETER_TUNING = True

## Load the GDSC2 response dataset

In [ ]:
response_data = load_dataset("GDSC2", path_data=str(DATA_DIR), measure=RESPONSE_MEASURE)
print(f"{response_data.dataset_name}: {len(response_data)} responses")

## Select models from `MODEL_FACTORY`

Same baseline (`NaiveTissueDrugMeanPredictor`) as notebook 05, so the sanity-check
table is directly comparable across both notebooks once consolidated in Task 4.

In [ ]:
models = [MODEL_FACTORY["DIPK"], MODEL_FACTORY["DrugGNN"], MODEL_FACTORY["SimpleNeuralNetwork"]]
baselines = [MODEL_FACTORY["NaiveTissueDrugMeanPredictor"]]

print("Models:   ", [m.__name__ for m in models])
print("Baselines:", [b.__name__ for b in baselines])

## Step 1 — Sanity-check runtime on LCO only

Run this first and look at the printed time before running the remaining three
splits below — these models are meaningfully more expensive than the Task 2
baselines, especially `DrugGNN` and `SimpleNeuralNetwork` with their hyperparameter
grids.

In [ ]:
start = time.time()
drug_response_experiment(
    models=models,
    baselines=baselines,
    response_data=response_data,
    test_mode="LCO",
    n_cv_splits=N_CV_SPLITS,
    run_id=RUN_ID,
    path_out=str(RESULTS_DIR),
    path_data=str(DATA_DIR),
    hyperparameter_tuning=HYPERPARAMETER_TUNING,
    overwrite=False,
)
lco_elapsed = time.time() - start
print(f"\nLCO done in {lco_elapsed / 60:.1f} min "
      f"(~{lco_elapsed / 60 * 3:.0f} min more for the remaining 3 splits, rough estimate)")

## Step 2 — Remaining splits (LPO, LDO, LTO)

Only run this after checking the Step 1 timing looks reasonable for your session
budget. Fold sizes differ slightly across split types (see the Session 4 sanity-check
table from notebook 04), so the estimate above is rough, not exact.

In [ ]:
for test_mode in ["LPO", "LDO", "LTO"]:
    print(f"\n========== {test_mode} ==========")
    start = time.time()
    drug_response_experiment(
        models=models,
        baselines=baselines,
        response_data=response_data,
        test_mode=test_mode,
        n_cv_splits=N_CV_SPLITS,
        run_id=RUN_ID,
        path_out=str(RESULTS_DIR),
        path_data=str(DATA_DIR),
        hyperparameter_tuning=HYPERPARAMETER_TUNING,
        overwrite=False,
    )
    elapsed = time.time() - start
    print(f"{test_mode} done in {elapsed / 60:.1f} min")

## Quick sanity check: per-fold Pearson r

Same smoke test as notebook 05 — full consolidation across all 5 models x 4
splits happens in Task 4.

In [ ]:
rows = []
for test_mode in ["LPO", "LCO", "LDO", "LTO"]:
    mode_dir = RESULTS_DIR / RUN_ID / response_data.dataset_name / test_mode
    for pred_file in sorted(glob(str(mode_dir / "*" / "predictions" / "predictions_split_*.csv"))):
        model_name = Path(pred_file).parts[-3]
        fold = int(Path(pred_file).stem.rsplit("_", 1)[-1])
        df = pd.read_csv(pred_file).dropna(subset=["response", "predictions"])
        r, _ = pearsonr(df["predictions"], df["response"])
        rows.append({"split": test_mode, "model": model_name, "fold": fold, "n": len(df), "pearson_r": round(r, 4)})

summary = pd.DataFrame(rows)
summary.groupby(["split", "model"])["pearson_r"].agg(["mean", "std", "count"])